In [38]:
from pathlib import Path
import csv
import json
import shutil
from collections import Counter

LAST_NAME = "Rojas"
STUDENT_ID = "TUPM-26-1244"

SEED_DIGIT = int(STUDENT_ID[-1])
ID_SUM = sum(int(d) for d in STUDENT_ID if d.isdigit())
NAME_LENGTH = len(LAST_NAME)

RECORD_LIMIT = 1000 + ID_SUM
TARGET_CLASS = (SEED_DIGIT % 4) + 2

print("Seed:", SEED_DIGIT)
print("ID Sum:", ID_SUM)
print("Records to process:", RECORD_LIMIT)
print("Target status class:", f"{TARGET_CLASS}")

Seed: 4
ID Sum: 19
Records to process: 1019
Target status class: 2


In [47]:
from pathlib import Path

WORK_DIR = Path.cwd()
DATA_DIR = WORK_DIR / "data"
OUTPUT_DIR = WORK_DIR / "output"
BACKUP_DIR = WORK_DIR / "backup"

for folder in (DATA_DIR, OUTPUT_DIR, BACKUP_DIR):
    folder.mkdir(exist_ok=True)

print("Notebook is running in:")
print(WORK_DIR.resolve())

print("\nFolders and files in current directory:")
for item in WORK_DIR.iterdir():
    print(item.name)

print("\nFiles inside data folder:")
if DATA_DIR.exists():
    for item in DATA_DIR.iterdir():
        print(item.name)

log_path = DATA_DIR / "weblog.csv"
print("\nDataset found:", log_path.exists())
print("Expected location:", log_path.resolve())

Notebook is running in:
C:\Users\rojas\Lab6_File_Handling_Rojas\Lab6_File_Handling_Additional_Exercises

Folders and files in current directory:
backup
data
Exercise1_Web_Log_Analysis.ipynb
output

Files inside data folder:
expanded_user_behavior_dataset.csv
Global_Cybersecurity_Threats_2015-2024.csv
weblog.csv

Dataset found: True
Expected location: C:\Users\rojas\Lab6_File_Handling_Rojas\Lab6_File_Handling_Additional_Exercises\data\weblog.csv


In [48]:

from collections import Counter
import csv

records_processed = 0
invalid_records = 0
status_counts = Counter()
url_counts = Counter()
selected_records = []

with open(log_path, "r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)

    for row in reader:
        if records_processed >= RECORD_LIMIT:
            break

        records_processed += 1

        ip = (row.get("IP") or "").strip()
        timestamp = (row.get("Time") or "").strip()
        url = (row.get("URL") or "").strip()
        status_text = (
            row.get("Staus") or row.get("Status") or ""
        ).strip()

        try:
            status = int(status_text)
        except ValueError:
            invalid_records += 1
            continue

        if not ip or not url:
            invalid_records += 1
            continue

        status_counts[status] += 1

        if status // 100 == TARGET_CLASS:
            selected_records.append([ip, timestamp, url, status])
            url_counts[url] += 1

top_urls = url_counts.most_common(5)

print("Records processed:", records_processed)
print("Invalid/missing records:", invalid_records)
print("Records in target class:", len(selected_records))
print("Count per status code:", dict(sorted(status_counts.items())))

print("Five most requested URLs:")
for url, count in top_urls:
    print(f"  {count:>4}  {url}")


Records processed: 1019
Invalid/missing records: 0
Records in target class: 850
Count per status code: {200: 842, 206: 8, 302: 143, 304: 21, 404: 5}
Five most requested URLs:
   104  GET /login.php HTTP/1.1
    91  GET /home.php HTTP/1.1
    38  GET /css/normalize.css HTTP/1.1
    37  GET /css/bootstrap.min.css HTTP/1.1
    37  GET /js/vendor/jquery-1.12.0.min.js HTTP/1.1


In [49]:
filtered_csv = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_web_filtered.csv"
json_path = OUTPUT_DIR / f"{LAST_NAME}_{STUDENT_ID}_web_summary.json"
backup_path = BACKUP_DIR / json_path.name

with open(filtered_csv, "w", encoding="utf-8", newline="") as file:
    writer = csv.writer(file)
    writer.writerow(["IP", "Time", "URL", "Status"])
    writer.writerows(selected_records)

summary = {
    "last_name": LAST_NAME,
    "student_id": STUDENT_ID,
    "records_processed": records_processed,
    "invalid_records": invalid_records,
    "target_status_class": f"{TARGET_CLASS}xx",
    "records_in_target_class": len(selected_records),
    "count_per_status_code": {str(k): v for k, v in sorted(status_counts.items())},
    "top_5_urls": [{"url": u, "count": c} for u, c in top_urls],
}

with open(json_path, "w", encoding="utf-8") as file:
    json.dump(summary, file, indent=4)

shutil.copy2(json_path, backup_path)
backup_ok = backup_path.exists() and backup_path.stat().st_size == json_path.stat().st_size

print("Filtered CSV file:", filtered_csv)
print("JSON summary file:", json_path)
print("Backup verification:", "VERIFIED" if backup_ok else "FAILED", "-", backup_path)

Filtered CSV file: c:\Users\rojas\Lab6_File_Handling_Rojas\Lab6_File_Handling_Additional_Exercises\output\Rojas_TUPM-26-1244_web_filtered.csv
JSON summary file: c:\Users\rojas\Lab6_File_Handling_Rojas\Lab6_File_Handling_Additional_Exercises\output\Rojas_TUPM-26-1244_web_summary.json
Backup verification: VERIFIED - c:\Users\rojas\Lab6_File_Handling_Rojas\Lab6_File_Handling_Additional_Exercises\backup\Rojas_TUPM-26-1244_web_summary.json


In [50]:
print("=== FINAL OUTPUT: WEB LOG ANALYSIS ===")
print("Student:", LAST_NAME, STUDENT_ID)
print("Records processed:", records_processed)
print("Target status class:", f"{TARGET_CLASS}xx")
print("Records in target class:", len(selected_records))
print("Count per status code:", dict(sorted(status_counts.items())))
print("Five most requested URLs:", top_urls)
print("Filtered CSV file:", filtered_csv.name)
print("JSON summary file:", json_path.name)
print("Backup verification:", "VERIFIED" if backup_ok else "FAILED")

=== FINAL OUTPUT: WEB LOG ANALYSIS ===
Student: Rojas TUPM-26-1244
Records processed: 1019
Target status class: 2xx
Records in target class: 850
Count per status code: {200: 842, 206: 8, 302: 143, 304: 21, 404: 5}
Five most requested URLs: [('GET /login.php HTTP/1.1', 104), ('GET /home.php HTTP/1.1', 91), ('GET /css/normalize.css HTTP/1.1', 38), ('GET /css/bootstrap.min.css HTTP/1.1', 37), ('GET /js/vendor/jquery-1.12.0.min.js HTTP/1.1', 37)]
Filtered CSV file: Rojas_TUPM-26-1244_web_filtered.csv
JSON summary file: Rojas_TUPM-26-1244_web_summary.json
Backup verification: VERIFIED


In [51]:

import csv

with open(log_path, "r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)

    print("CSV COLUMN NAMES:")
    print(reader.fieldnames)

    print("\nFIRST 3 RECORDS:")
    for i, row in enumerate(reader):
        print(row)
        if i == 2:
            break


CSV COLUMN NAMES:
['IP', 'Time', 'URL', 'Staus']

FIRST 3 RECORDS:
{'IP': '10.128.2.1', 'Time': '[29/Nov/2017:06:58:55', 'URL': 'GET /login.php HTTP/1.1', 'Staus': '200'}
{'IP': '10.128.2.1', 'Time': '[29/Nov/2017:06:59:02', 'URL': 'POST /process.php HTTP/1.1', 'Staus': '302'}
{'IP': '10.128.2.1', 'Time': '[29/Nov/2017:06:59:03', 'URL': 'GET /home.php HTTP/1.1', 'Staus': '200'}


In [52]:
print("Dataset found:", log_path.exists())
print("Dataset location:", log_path.resolve())

Dataset found: True
Dataset location: C:\Users\rojas\Lab6_File_Handling_Rojas\Lab6_File_Handling_Additional_Exercises\data\weblog.csv
